# Unified BIDSme notebook for IronSleep / TerraX / LORAKS

This notebook replaces the separate normal and LORAKS notebooks.

It is designed to work with the generalized plugins:

- `plugin_prepare_auto_tempfolder_no_idinfo_commented_nk.py`
- `plugin_bidsify_auto_with_csv_tsv_nk.py`

The intended boundary is:

- **prepare** writes only temporary mappings inside the prepared/temp folder, e.g. `.prepare_id_map/`
- **bidsify** writes durable `id_info/*.csv`, `sub-*_sessions.tsv`, and `sub-*_sessions.json`
- the same bidsify plugin is used for normal/TerraX/dcm2niix data and LORAKS derivatives


## 1. Imports and user settings

Usually you only need to edit this first code cell.

`STREAMS_TO_RUN = "auto"` means:

- run the standard stream if standard input data are detected
- run the LORAKS stream if LORAKS input data are detected
- run both if both are detected

Use `STREAMS_TO_RUN = ["standard"]` or `["loraks"]` to force one stream.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess

# -----------------------------
# Main project paths
# -----------------------------

DATASET_PATH = Path("/data/pt_03187/data/in_vivo/")

SOURCE_PATH = DATASET_PATH / "source"

# Standard BIDS output:
STANDARD_PREPARED_PATH = DATASET_PATH / "temp"
STANDARD_BIDS_PATH = DATASET_PATH / "bids"

# LORAKS derivative output:
LORAKS_PREPARED_PATH = DATASET_PATH / "temp" / "LORAKS"
LORAKS_BIDS_PATH = DATASET_PATH / "bids" / "derivatives" / "LORAKS"

WORKING_DIR = Path.cwd()

# -----------------------------
# Generalized plugin paths
# -----------------------------

PREPARE_PLUGIN = WORKING_DIR / "plugins_bidsme" / "plugin_prepare_auto_tempfolder_no_idinfo_commented_nk.py"
BIDSIFY_PLUGIN = WORKING_DIR / "plugins_bidsme" / "plugin_bidsify_auto_with_csv_tsv_nk.py"

PARTICIPANTS_TEMPLATE = WORKING_DIR / "supplementary" / "table_templates" / "participants_nk.json"
SESSIONS_TEMPLATE = WORKING_DIR / "supplementary" / "table_templates" / "sessions_nk.json"

# -----------------------------
# Processing selection
# -----------------------------

# "auto" = detect streams automatically.
# Alternatives: ["standard"], ["loraks"], or ["standard", "loraks"]
STREAMS_TO_RUN = "auto"

# Optional subject restriction.
# Use BIDS-style labels after prepare mapping if your source folders are already BIDS-like.
# Set to None to process all subjects.
SUB_LIST = ["sub-007"]
# SUB_LIST = None

# Whether LORAKS sensitivity maps should be included.
# False is safest unless your bidsmap explicitly expects them.
INCLUDE_SMAPS = False

# If True, bidsify command includes --skip-existing.
SKIP_EXISTING = False


## 2. Basic path checks


In [ ]:
paths_to_check = {
    "Dataset": DATASET_PATH,
    "Source": SOURCE_PATH,
    "Working directory": WORKING_DIR,
    "Prepare plugin": PREPARE_PLUGIN,
    "Bidsify plugin": BIDSIFY_PLUGIN,
    "Participants template": PARTICIPANTS_TEMPLATE,
    "Sessions template": SESSIONS_TEMPLATE,
}

for name, path in paths_to_check.items():
    print(f"{name:22s}: {path} -> {path.exists()}")


## 3. Detect available input streams

This cell only inspects folder names and filenames.

The detection is intentionally simple:

- **standard** data are assumed if common folders such as `dcm2niix/`, `nii/`, or `nii_dcm2niix/` are present
- **LORAKS** data are assumed if `nii_loraks_recon/` exists or filenames contain `rec-loraks`


In [ ]:
def path_contains_name(root: Path, names):
    """Return True if any folder/file below root has one of the requested names."""
    if not root.exists():
        return False

    names = {name.casefold() for name in names}

    for path in root.rglob("*"):
        if path.name.casefold() in names:
            return True

    return False


def path_contains_text(root: Path, text):
    """Return True if any path below root contains `text` in its name/path."""
    if not root.exists():
        return False

    text = text.casefold()

    for path in root.rglob("*"):
        if text in str(path).casefold():
            return True

    return False


def detect_streams(source_path: Path):
    """Detect whether standard data, LORAKS data, or both are present."""
    has_standard = (
        path_contains_name(source_path, ["dcm2niix", "nii", "nii_dcm2niix"])
        or path_contains_text(source_path, "dcm2niix")
    )

    has_loraks = (
        path_contains_name(source_path, ["nii_loraks_recon"])
        or path_contains_text(source_path, "rec-loraks")
        or path_contains_text(source_path, "loraks")
    )

    streams = []
    if has_standard:
        streams.append("standard")
    if has_loraks:
        streams.append("loraks")

    return streams


detected_streams = detect_streams(SOURCE_PATH)

if STREAMS_TO_RUN == "auto":
    streams_to_run = detected_streams
else:
    streams_to_run = list(STREAMS_TO_RUN)

print("Detected streams:", detected_streams)
print("Streams selected:", streams_to_run)

if not streams_to_run:
    raise RuntimeError(
        "No input stream was selected. Check SOURCE_PATH or set STREAMS_TO_RUN manually."
    )


## 4. Define per-stream settings

Each stream gets its own prepared path, output path, and `data_dirs`.

The same generalized plugins are used for both streams.


In [ ]:
STREAM_CONFIG = {
    "standard": {
        "prepared_path": STANDARD_PREPARED_PATH,
        "bids_path": STANDARD_BIDS_PATH,
        # This mirrors the old TerraX/dcm2niix notebook.
        # If your source uses a different layout, edit this mapping.
        "data_dirs": {
            "dcm2niix/*": "MRI",
            # Useful alternatives:
            # "nii/*": "MRI",
            # "nii_dcm2niix/*": "MRI",
        },
        "include_smaps": INCLUDE_SMAPS,
    },
    "loraks": {
        "prepared_path": LORAKS_PREPARED_PATH,
        "bids_path": LORAKS_BIDS_PATH,
        # This mirrors the old LORAKS notebook.
        "data_dirs": {
            "nii_loraks_recon": "MRI",
        },
        "include_smaps": INCLUDE_SMAPS,
    },
}

for stream in streams_to_run:
    cfg = STREAM_CONFIG[stream]
    print(f"\n[{stream}]")
    print("prepared_path:", cfg["prepared_path"])
    print("bids_path:    ", cfg["bids_path"])
    print("data_dirs:    ", cfg["data_dirs"])
    print("include_smaps:", cfg["include_smaps"])


## 5. Initialize BIDSme


In [ ]:
import bidsme

logger = bidsme.init()
logger.setLevel("INFO")


## 6. Prepare

Important: with the generalized prepare plugin, preparation may create temporary mapping files inside the prepared folder, e.g.

```text
<prepared_path>/.prepare_id_map/
```

It should not write final `id_info/`, `sessions.tsv`, or `sessions.json`.


In [ ]:
def prepare_stream(stream: str):
    cfg = STREAM_CONFIG[stream]

    print(f"\n=== PREPARE: {stream} ===")
    print("source:  ", SOURCE_PATH)
    print("prepared:", cfg["prepared_path"])

    kwargs = dict(
        data_dirs=cfg["data_dirs"],
        plugin_file=str(PREPARE_PLUGIN),
        part_template=str(PARTICIPANTS_TEMPLATE),
        plugin_opt={
            "include_smaps": cfg["include_smaps"],
        },
    )

    if SUB_LIST:
        kwargs["sub_list"] = SUB_LIST

    bidsme.prepare(str(SOURCE_PATH), str(cfg["prepared_path"]), **kwargs)

    bidsme.tools.info.reporterrors(logger)
    bidsme.tools.info.reseterrors(logger)


for stream in streams_to_run:
    prepare_stream(stream)


## 7. Create or update `bidsmap.yaml`

Run this cell, inspect/fix the generated `bidsmap.yaml`, then re-run until the reported errors/warnings are acceptable.

The bidsmap is written under:

```text
<bids_path>/code/bidsme/bidsmap.yaml
```


In [ ]:
def map_stream(stream: str):
    cfg = STREAM_CONFIG[stream]

    print(f"\n=== MAP: {stream} ===")
    print("prepared:", cfg["prepared_path"])
    print("bids:    ", cfg["bids_path"])

    kwargs = dict(
        plugin_file=str(BIDSIFY_PLUGIN),
        plugin_opt={
            "bidsmap_step": True,
            "include_smaps": cfg["include_smaps"],
            "sessions_tsv_template": str(SESSIONS_TEMPLATE),
        },
    )

    if SUB_LIST:
        kwargs["sub_list"] = SUB_LIST

    bidsme.mapper(str(cfg["prepared_path"]), str(cfg["bids_path"]), **kwargs)

    bidsme.tools.info.reporterrors(logger)
    bidsme.tools.info.reseterrors(logger)


for stream in streams_to_run:
    map_stream(stream)


## 8. Bidsify

This calls the BIDSme CLI because that was the structure of the original notebooks.

The generalized bidsify plugin should now write/merge:

```text
<bids_path>/id_info/subject_ids.csv
<bids_path>/id_info/<sub>_sessions.csv
<bids_path>/sub-XXX/sub-XXX_sessions.tsv
<bids_path>/sub-XXX/sub-XXX_sessions.json
```

Existing rows should be updated/appended rather than blindly overwritten.


In [ ]:
def run_shell(cmd):
    print(" ".join(shlex.quote(str(part)) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)


def bidsify_stream(stream: str):
    cfg = STREAM_CONFIG[stream]
    map_file = cfg["bids_path"] / "code" / "bidsme" / "bidsmap.yaml"

    print(f"\n=== BIDSIFY: {stream} ===")
    print("prepared:", cfg["prepared_path"])
    print("bids:    ", cfg["bids_path"])
    print("bidsmap: ", map_file)

    cmd = [
        "bidsme", "bidsify",
        str(cfg["prepared_path"]),
        str(cfg["bids_path"]),
        "-b", str(map_file),
        "--plugin", str(BIDSIFY_PLUGIN),
        "-o", f"include_smaps={cfg['include_smaps']}",
        "-o", f"sessions_tsv_template={SESSIONS_TEMPLATE}",
    ]

    if SUB_LIST:
        cmd += ["--participants"] + list(SUB_LIST)

    if SKIP_EXISTING:
        cmd += ["--skip-existing"]

    run_shell(cmd)


for stream in streams_to_run:
    bidsify_stream(stream)


## 9. Quick output checks

This cell does not validate BIDS. It only checks that the expected bookkeeping outputs exist.


In [ ]:
for stream in streams_to_run:
    cfg = STREAM_CONFIG[stream]
    bids_path = cfg["bids_path"]

    print(f"\n=== CHECK: {stream} ===")
    print("BIDS path exists:", bids_path.exists())

    id_info = bids_path / "id_info"
    print("id_info exists:", id_info.exists())
    if id_info.exists():
        print("id_info files:")
        for path in sorted(id_info.glob("*.csv")):
            print("  ", path.relative_to(bids_path))

    print("sessions.tsv files:")
    for path in sorted(bids_path.glob("sub-*/sub-*_sessions.tsv")):
        print("  ", path.relative_to(bids_path))


## 10. Notes

The old notebooks differed mainly in three places:

| Step | Standard/TerraX notebook | LORAKS notebook | Unified notebook |
|---|---|---|---|
| prepared path | `temp` | `temp/LORAKS` | selected by stream |
| BIDS output | `bids` | `bids/derivatives/LORAKS` | selected by stream |
| plugin files | TerraX/normal plugins | LORAKS plugins | generalized prepare + generalized bidsify |

If auto-detection chooses the wrong stream, set `STREAMS_TO_RUN` manually in the first code cell.
